# South Florida Traffic Crash Analysis (2021–2023)
**Counties:** Miami-Dade | Broward | Palm Beach  
**Dataset:** 500 records × 8 features  
**Tools:** Python, pandas, matplotlib, seaborn, scikit-learn


---
## Step 3 — Understand the Data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_excel('data/florida_crashes_cleaned.xlsx', sheet_name='Cleaned Data')

print('=== SHAPE ===')
print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')

print('\n=== COLUMN TYPES ===')
print(df.dtypes)

print('\n=== FIRST 5 ROWS ===')
df.head()


In [ ]:
print('=== NULL VALUES ===')
print(df.isnull().sum())

print('\n=== UNIQUE VALUES PER COLUMN ===')
for col in df.columns:
    print(f'{col}: {sorted(df[col].dropna().unique().tolist())[:10]}')


In [ ]:
print('=== NUMERIC STATS ===')
df['accidents_count'].describe()


---
## Step 4 — Clean the Data


In [ ]:
# All checks — document results
print('CLEANING LOG')
print('='*40)
print(f'Original rows:     {len(df)}')
print(f'Null values:       {df.isnull().sum().sum()}')
print(f'Duplicate rows:    {df.duplicated().sum()}')
print(f'County values:     {sorted(df["county"].unique())}')
print(f'Year values:       {sorted(df["year"].unique())}')
print(f'Day values:        {sorted(df["day"].unique())}')
print(f'Injury values:     {sorted(df["injury"].unique())}')
print(f'Min accidents:     {df["accidents_count"].min()}')
print(f'Max accidents:     {df["accidents_count"].max()}')
print('\nRESULT: All checks passed. No cleaning actions required.')
print(f'Final rows:        {len(df)}')


---
## Step 5 — Transform / Feature Engineering


In [ ]:
# Day order for correct chart sorting
df['day_order'] = df['day'].map({'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7})

# Weekday vs Weekend
df['day_type'] = df['day'].apply(lambda x: 'Weekend' if x in ['Sat','Sun'] else 'Weekday')

# Readable time labels
time_labels = {
    '0-4':'Late Night', '5-8':'Early Morning', '9-12':'Mid Morning',
    '13-16':'Afternoon', '17-20':'Evening Rush', '21-23':'Night'
}
df['time_label'] = df['time_range'].map(time_labels)

# Numeric hour start for sorting
df['hour_start'] = df['time_range'].str.split('-').str[0].astype(int)

# Binary injury for modeling
df['injury_binary'] = (df['injury'] == 'Yes').astype(int)

# High accident flag (above 75th percentile)
threshold = df['accidents_count'].quantile(0.75)
df['high_count_flag'] = df['accidents_count'].apply(lambda x: 'High' if x >= threshold else 'Normal')

print(f'New columns added: {["day_order","day_type","time_label","hour_start","injury_binary","high_count_flag"]}')
print(f'75th percentile threshold for High flag: {threshold}')
df.head(3)


---
## Step 6 — Exploratory Data Analysis (EDA)


In [ ]:
print('=== TOTAL ACCIDENTS BY COUNTY ===')
print(df.groupby('county')['accidents_count'].sum().sort_values(ascending=False))

print('\n=== TOTAL ACCIDENTS BY YEAR ===')
print(df.groupby('year')['accidents_count'].sum())

print('\n=== AVG ACCIDENTS BY REASON ===')
print(df.groupby('reason')['accidents_count'].mean().sort_values(ascending=False).round(2))

print('\n=== AVG ACCIDENTS BY TIME RANGE ===')
print(df.groupby('time_range')['accidents_count'].mean().sort_values(ascending=False).round(2))

print('\n=== AVG ACCIDENTS BY AGE GROUP ===')
print(df.groupby('age_group')['accidents_count'].mean().sort_values(ascending=False).round(2))


In [ ]:
print('=== COUNTY x YEAR TOTALS ===')
print(df.groupby(['year','county'])['accidents_count'].sum().unstack())

print('\n=== INJURY RATE BY REASON ===')
inj_rate = df.groupby('reason').agg(
    total=('injury_binary','count'),
    injured=('injury_binary','sum')
)
inj_rate['injury_rate_%'] = (inj_rate['injured']/inj_rate['total']*100).round(1)
print(inj_rate[['injured','total','injury_rate_%']].sort_values('injury_rate_%',ascending=False))


---
## Step 7 — Analyze & Model


In [ ]:
# Weekday vs Weekend
print('=== WEEKDAY VS WEEKEND ===')
print(df.groupby('day_type')['accidents_count'].agg(['mean','sum','count']).round(2))

# Year-over-year change
print('\n=== YEAR OVER YEAR CHANGE ===')
trend = df.groupby(['county','year'])['accidents_count'].sum().unstack()
trend['change_2021_2023_%'] = ((trend[2023]-trend[2021])/trend[2021]*100).round(1)
print(trend)


In [ ]:
# Random Forest model to predict injury
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df_model = df.copy()
features = ['county','reason','age_group','time_range','day_type']
for col in features:
    df_model[col] = LabelEncoder().fit_transform(df_model[col])

X = df_model[features + ['accidents_count']]
y = df_model['injury_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

print(f'Model Accuracy: {clf.score(X_test, y_test):.2%}')
print('\nClassification Report:')
print(classification_report(y_test, clf.predict(X_test)))

print('\n=== FEATURE IMPORTANCE (what predicts injury most) ===')
fi = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(fi.round(4))


---
## Step 8 — Visualize
*All charts saved to the `charts/` folder*


In [ ]:
BG='#0F1117'; CARD='#1A1D27'; CARD2='#22263A'; TEXT='#E8EAF0'
MUTED='#7A8099'; ACC='#F4A118'; RED='#E05B2E'; TEAL='#1ECFB0'; BLUE='#4D9CFF'
C_CTY={'Miami-Dade':'#4D9CFF','Broward':'#F4A118','Palm Beach':'#1ECFB0'}
C_RSN={'Drowsy':'#E05B2E','Speeding':'#F4A118','DUI':'#4D9CFF',
       'Distracted Driving':'#1ECFB0','Weather':'#8899BB'}

def sax(ax):
    ax.set_facecolor(CARD); ax.tick_params(colors=MUTED, labelsize=9)
    for lbl in ax.get_xticklabels()+ax.get_yticklabels(): lbl.set_color(MUTED)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    ax.title.set_color(TEXT)
    for sp in ax.spines.values(): sp.set_edgecolor(CARD2)
    ax.grid(axis='y',color=CARD2,linewidth=0.6,alpha=0.9); ax.set_axisbelow(True)

print('Style settings defined. See charts/ folder for all 9 saved charts.')
print('Charts included:')
for i,name in enumerate([
    'chart1_county_year.png     — Grouped bar: accidents by county per year',
    'chart2_avg_by_reason.png   — Horizontal bar: avg accidents by cause',
    'chart3_heatmap_day_time.png — Heatmap: day × time of day',
    'chart4_injury_by_reason.png — Stacked bar: injury vs no injury by cause',
    'chart5_trend_line.png      — Line chart: year trend per county',
    'chart6_age_group.png       — Bar: avg accidents by age group',
    'chart7_donut_reason.png    — Donut: share of accidents by cause',
    'chart8_boxplot_county.png  — Box plot: distribution by county',
    'chart9_weekday_weekend.png — Grouped bar: weekday vs weekend'
],1): print(f'  {i}. {name}')


---
## Step 9 — Key Findings & Recommendations


In [ ]:
print('KEY FINDINGS')
print('='*55)
print()
print('Finding 1 — 2023 worst year')
trend2 = df.groupby(['county','year'])['accidents_count'].sum().unstack()
for county in ['Broward','Miami-Dade','Palm Beach']:
    chg = (trend2.loc[county,2023]-trend2.loc[county,2021])/trend2.loc[county,2021]*100
    print(f'  {county}: {chg:+.1f}% change from 2021 to 2023')
print()
print('Finding 2 — Most dangerous time')
avg_time = df.groupby('time_range')['accidents_count'].mean()
print(f'  Highest avg: {avg_time.idxmax()} = {avg_time.max():.1f}')
print(f'  Lowest avg:  {avg_time.idxmin()} = {avg_time.min():.1f}')
print()
print('Finding 3 — Top cause')
print(f'  {df.groupby("reason")["accidents_count"].mean().idxmax()} = highest avg accidents')
print()
print('Finding 4 — Highest risk age group')
print(f'  {df.groupby("age_group")["accidents_count"].mean().idxmax()} = highest avg accidents')
print()
print('Finding 5 — Overall injury rate')
print(f'  {df["injury_binary"].mean()*100:.1f}% of all records resulted in injury')


---
## Recommendations

1. **Increase DUI checkpoints and drowsy-driving campaigns** during early morning hours (5–8 AM) — the highest-risk window across all counties.

2. **Target the 25–34 age group** with road safety campaigns through digital and social media — they show the highest average accident count.

3. **Prioritize enforcement in Broward and Miami-Dade** for 2024, as both show accelerating trends. Broward showed the steepest rise from 2021 to 2023.

4. **Invest in drowsy driving awareness programs** — Drowsy driving ranks as the top cause by average accident count, above even DUI.
